# TXF session-flow persistence v0.1.0
Live FinLab data, immutable Google Drive raw snapshots, append-only research results.
**Not a trading signal. Do not confuse net trading contracts with OI change.**

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

REPOSITORY_URL = "https://github.com/hh4832/txf_session_flow_persistence.git"
repo_dir = Path("/content/txf_session_flow_persistence")

# Never mistake an existing checkout for an up-to-date checkout.
if repo_dir.exists():
    if not (repo_dir / ".git").is_dir():
        raise RuntimeError(f"Existing path is not a Git checkout: {repo_dir}")
    origin = subprocess.run(
        ["git", "-C", str(repo_dir), "remote", "get-url", "origin"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Unexpected Git remote in {repo_dir}: {origin}")
    if subprocess.run(
        ["git", "-C", str(repo_dir), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip():
        raise RuntimeError("Git checkout contains local changes; refusing to overwrite them")
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "merge", "--ff-only", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(repo_dir)], check=True)

# Use the CURRENT NOTEBOOK KERNEL's interpreter (not a separate Python install).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[finlab,drive]"],
    cwd=repo_dir, check=True,
)

# Editable installs may add a .pth file that the already-running Colab kernel
# has not processed yet. Also /content/<repo-name> can resolve as an empty
# namespace package instead of the real src-layout Python package.
src_dir = repo_dir / "src"
if not (src_dir / "txf_session_flow_persistence" / "config.py").is_file():
    raise RuntimeError("Repository is missing src/txf_session_flow_persistence/config.py")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
for name in list(sys.modules):
    if name == "txf_session_flow_persistence" or name.startswith("txf_session_flow_persistence."):
        del sys.modules[name]
importlib.invalidate_caches()
config_module = importlib.import_module("txf_session_flow_persistence.config")
expected_config = (src_dir / "txf_session_flow_persistence" / "config.py").resolve()
if Path(config_module.__file__).resolve() != expected_config:
    raise RuntimeError(f"Wrong Python module imported: {config_module.__file__}")
commit_sha = subprocess.run(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Research checkout: {commit_sha[:12]}")
print(f"Python package: {config_module.__file__}")

## Authenticate
Uses Colab Secrets named `FINLAB_API_TOKEN` if data is not already cached. Do not print the value.
Drive OAuth authentication grants the notebook access to your pre-existing folders.

In [ ]:
import os
from google.colab import auth, userdata
auth.authenticate_user()
try:
    token = userdata.get("FINLAB_API_TOKEN")
    if token:
        os.environ["FINLAB_API_TOKEN"] = token
except Exception:
    print("No token loaded; this is okay only when both raw snapshots already exist.")

## Run research
Set END_DATE to the last exchange trade date that both FinLab feeds have published.
Default column label: 臺股期貨_外資及陸資 — change INSTRUMENT if your FinLab table uses another exact label.
`REFRESH` must be True only to explicitly create a NEW immutable revision; never overwrites earlier snapshots.

In [ ]:
from txf_session_flow_persistence.config import RAW_FOLDER_ID, RESULTS_FOLDER_ID, ResearchConfig
from txf_session_flow_persistence.storage import GoogleDriveStore
from txf_session_flow_persistence.runner import run
START_DATE = "2023-10-16"
END_DATE = "2026-10-22"
INSTRUMENT = "臺股期貨_外資及陸資"
REFRESH = False
cfg = ResearchConfig(start=START_DATE, end=END_DATE, instrument=INSTRUMENT).validate()
manifest, tables = run(GoogleDriveStore(RAW_FOLDER_ID, RESULTS_FOLDER_ID), cfg, refresh=REFRESH)
print({k: manifest[k] for k in ["status", "run_name", "full_trading_dates", "valid_paired_dates", "unusable_dates"]})
display(tables["direction_summary"])